<a href="https://colab.research.google.com/github/ronk83952-cmd/Mist/blob/main/ollama_gemma4_E4B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Ollama and dependencies

In [1]:
import subprocess
import sys

# Install zstd
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-zstd'], check=True)
    print("python-zstd installed successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error installing python-zstd: {e}")
    print("Attempting to install apt package for zstd...")
    try:
        subprocess.run(['apt-get', 'update'], check=True)
        subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
        print("zstd apt package installed successfully.")
    except subprocess.CalledProcessError as e_apt:
        print(f"Error installing zstd via apt: {e_apt}")
        print("Please ensure zstd is available on your system.")

Error installing python-zstd: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'python-zstd']' returned non-zero exit status 1.
Attempting to install apt package for zstd...
zstd apt package installed successfully.


### Download and install Ollama

In [2]:
import os

# Download the Ollama installation script
!curl -fsSL https://ollama.com/install.sh -o install_ollama.sh

# Make the script executable
!chmod +x install_ollama.sh

# Run the installation script. This will install Ollama to /usr/local/bin
# Note: This might require root privileges in some environments.
# In Colab, it often works without explicit sudo for user-level installs.
!./install_ollama.sh

print("Ollama installation script executed. Verifying installation...")

# Verify installation
try:
    result = subprocess.run(['ollama', '--version'], capture_output=True, text=True, check=True)
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Ollama not found or error during version check: {e}")
    print(f"Stderr: {e.stderr}")
except FileNotFoundError:
    print("Ollama command not found. Installation might have failed or it's not in PATH.")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installation script executed. Verifying installation...



### Run Ollama

In [3]:
# Start the Ollama server in the background
# The 'nohup' command allows the process to continue running after the terminal is closed.
# The '&' puts the process in the background.
# The output is redirected to ollama_output.log

!nohup ollama serve > ollama_output.log 2>&1 &

print("Ollama server started in the background. Check 'ollama_output.log' for logs.")
print("You can now interact with Ollama using `ollama` commands or its API.")

Ollama server started in the background. Check 'ollama_output.log' for logs.
You can now interact with Ollama using `ollama` commands or its API.


In [4]:
# Download the gemma4:4b model
print("Downloading gemma4:4b model...")
!ollama pull gemma4:e4b

In [5]:
# Run the gemma4:e4b model with the specified prompt
print("Running gemma4:e4b model...")
!ollama run gemma4:e4b "this is new model gemma4:e4b parameters"

Running gemma4:e4b model...
Thinking...
Thinking Process:

1.  **Analyze the input:** The user stated: "this is new model gemma4:e4b p
parameters". This looks like a declarative statement, possibly identifying 
a model or its parameters.
2.  **Identify the core topic:** The topic is "Gemma 4" and a specific size
size/parameter designation ("e4b parameters").
3.  **Determine the appropriate response:** Since the user is *telling* me 
this information, I should acknowledge it and provide a neutral, helpful co
confirmation or general statement about the model family, aligning with my 
established persona (Gemma 4).
4.  **Self-Correction/Refinement (Gemma Identity):** I need to ensure my re
response is consistent with being the Gemma 4 model. I should acknowledge t
the model family while staying factual and helpful.
5.  **Formulate the response:** Acknowledge the information and offer assis
assistance.

    *   *Draft 1 (Internal):* Okay, Gemma 4 e4b. How can I help you with it
it? (Too br

### Set up Gradio for Chat Interface

In [6]:
# Install Gradio
%pip install --upgrade gradio ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0


In [7]:
import gradio as gr
import ollama

def chat_with_ollama(message, history):
    try:
        # Ensure the model is available before trying to run it
        # You might want to check for a specific model, e.g., 'gemma:2b'
        # For now, we'll assume gemma4:26b is the desired model from previous steps
        response = ollama.chat(model='gemma4:e4b', messages=[
            {'role': 'user', 'content': message}
        ])
        return response['message']['content']
    except Exception as e:
        return f"Error communicating with Ollama: {e}. Please ensure the model is pulled and the Ollama server is running."

In [8]:
# Create and launch the Gradio interface
# Set share=True to get a public URL for the Gradio app
iface = gr.ChatInterface(chat_with_ollama,
                         title='Chat with gemma4:e4b (via Ollama)',
                         description='Type your message and press Enter to chat with the gemma4:e4b model.').queue()
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://941d5650aacbe01e03.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import IPython

# This JavaScript code will periodically click on a button to keep the Colab session alive.
# It's a common workaround to prevent Colab from timing out due to inactivity.
# Note: This might not work in all scenarios and is a browser-side workaround.
js_code = '''
  function KeepClicking() {
    console.log("Clicking button to keep Colab alive...");
    document.querySelector("colab-toolbar-button#connect").click();
  }
  setInterval(KeepClicking, 60 * 1000); // Click every 60 seconds
'''

display(IPython.display.Javascript(js_code))
print("Colab session will attempt to stay active. Look for 'Clicking button to keep Colab alive...' messages in your browser's developer console.")

<IPython.core.display.Javascript object>

Colab session will attempt to stay active. Look for 'Clicking button to keep Colab alive...' messages in your browser's developer console.
